In [1]:
import tensorflow as tf
from keras.utils import to_categorical
from keras import layers, models, optimizers, metrics, regularizers
import datetime

import sys
sys.path.append('..')
from scripts.prepare_data import download_data, preproces_without_oversampling, augment_to_target_counts
from sklearn.utils.class_weight import compute_class_weight

import numpy as np

2026-01-15 19:50:11.725185: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-01-15 19:50:11.755684: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-15 19:50:12.530942: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
/home/kuba/RNN-ECG-analysis/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning

In [2]:
# Pobieranie i wstępne przetwarzanie danych
df_base_preproc = download_data()
X_train, X_test, y_train, y_test = preproces_without_oversampling(df_base_preproc)

In [3]:
X_train_arr = X_train.values if hasattr(X_train, 'values') else X_train
X_test_arr = X_test.values if hasattr(X_test, 'values') else X_test
y_train_np = y_train.values if hasattr(y_train, 'values') else y_train

X_train_flat = X_train_arr
X_test_flat = X_test_arr

print(f"X_train_flat shape (dla Dense): {X_train_flat.shape}")

# Augmentacja mniejszościowych klas
X_train_3d = X_train_flat.reshape((X_train_flat.shape[0], X_train_flat.shape[1], 1))
X_train_aug_3d, y_train_aug = augment_to_target_counts(X_train_3d, y_train_np, {2: 16000, 3: 16000})

X_train_aug = X_train_aug_3d.reshape((X_train_aug_3d.shape[0], -1))

print(f"\nPo augmentacji:")
print(f"X_train_aug shape: {X_train_aug.shape}")
print(f"Rozkład klas: {np.unique(y_train_aug, return_counts=True)}")

perm = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[perm]
y_train_aug = y_train_aug[perm]

X_train_flat shape (dla Dense): (140578, 32)
Stan początkowy: {np.int64(0): np.int64(122837), np.int64(1): np.int64(16000), np.int64(2): np.int64(1566), np.int64(3): np.int64(175)}
Klasa 0: Brak celu w słowniku. Pozostaje bez zmian (122837).
Klasa 1: Brak celu w słowniku. Pozostaje bez zmian (16000).
Klasa 2: Augmentacja z 1566 do 16000 (+14434 próbek)
Klasa 3: Augmentacja z 175 do 16000 (+15825 próbek)

Po augmentacji:
X_train_aug shape: (170837, 32)
Rozkład klas: (array([0, 1, 2, 3]), array([122837,  16000,  16000,  16000]))


In [4]:
# Tworzenie datasetów
batch_size = 32

y_train_preproc = to_categorical(y_train_aug)
y_test_preproc = to_categorical(y_test)

train_ds = tf.data.Dataset.from_tensor_slices((X_train_aug, y_train_preproc)) \
    .shuffle(1000) \
    .batch(batch_size) \
    .prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_test_flat, y_test_preproc)) \
    .batch(batch_size) \
    .prefetch(tf.data.AUTOTUNE)

print(f"Train batches: {len(X_train_aug) // batch_size}")
print(f"Val batches: {len(X_test_flat) // batch_size}")

Train batches: 5338
Val batches: 1098


I0000 00:00:1768503019.997923  106464 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1218 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [5]:
dense_model = models.Sequential([
    layers.Input(shape=(X_train_aug.shape[1],)),
    layers.BatchNormalization(),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.L2()),
    layers.Dropout(0.3),
    layers.Dense(64, activation='relu', kernel_regularizer=regularizers.L2()),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax')
])

dense_model.compile(
    loss='categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=0.001),
    metrics=[metrics.F1Score(average='macro')]
)

log_dir = "../logs/fit/dense_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_f1_score', patience=5, restore_best_weights=True, mode='max')

history = dense_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[tensorboard_callback, early_stopping]
)

dense_model.save("models/dense_baseline.keras")

Epoch 1/30


2026-01-15 19:50:23.028380: I external/local_xla/xla/service/service.cc:163] XLA service 0x7fe320016420 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-01-15 19:50:23.028391: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 3060, Compute Capability 8.6
2026-01-15 19:50:23.045630: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-01-15 19:50:23.143077: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91701
2026-01-15 19:50:23.212680: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:50:23.212742: I e

 129/5339 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - f1_score: 0.3672 - loss: 2.1901

I0000 00:00:1768503024.984330  106621 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5324/5339 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - f1_score: 0.7294 - loss: 0.6654

2026-01-15 19:50:32.472753: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:50:32.890923: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_948', 24 bytes spill stores, 24 bytes spill loads

2026-01-15 19:50:32.954655: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_948', 20 bytes spill stores, 20 bytes spill loads



5339/5339 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - f1_score: 0.7296 - loss: 0.6648

2026-01-15 19:50:34.068110: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:50:34.381705: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_79', 100 bytes spill stores, 100 bytes spill loads



5339/5339 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - f1_score: 0.7797 - loss: 0.4347 - val_f1_score: 0.6986 - val_loss: 0.1706
Epoch 2/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - f1_score: 0.8230 - loss: 0.3106 - val_f1_score: 0.6845 - val_loss: 0.1710
Epoch 3/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - f1_score: 0.8313 - loss: 0.2937 - val_f1_score: 0.6623 - val_loss: 0.1690
Epoch 4/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - f1_score: 0.8369 - loss: 0.2856 - val_f1_score: 0.6731 - val_loss: 0.1580
Epoch 5/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - f1_score: 0.8417 - loss: 0.2800 - val_f1_score: 0.6839 - val_loss: 0.1464
Epoch 6/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - f1_score: 0.8453 - loss: 0.2767 - val_f1_score: 0.6979 - val_loss: 0.1384


In [6]:
# Większy model Dense
dense_larger = models.Sequential([
    layers.Input(shape=(X_train_aug.shape[1],)),
    layers.BatchNormalization(),
    layers.Dense(256, activation='relu', kernel_regularizer=regularizers.L2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(128, activation='relu', kernel_regularizer=regularizers.L2(0.001)),
    layers.Dropout(0.4),
    layers.Dense(64, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(4, activation='softmax')
])

dense_larger.compile(
    loss='categorical_crossentropy',
    optimizer=optimizers.Adam(learning_rate=0.001),
    metrics=[metrics.F1Score(average='macro')]
)

log_dir = "../logs/fit/dense_larger_" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = tf.keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=1)
early_stopping = tf.keras.callbacks.EarlyStopping(monitor='val_f1_score', patience=5, restore_best_weights=True, mode='max')

history = dense_larger.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=[tensorboard_callback, early_stopping]
)

dense_larger.save("models/dense_larger.keras")

Epoch 1/30


2026-01-15 19:55:49.265594: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:55:49.265616: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:55:49.265646: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:55:50.050080: I external/l

5305/5339 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step - f1_score: 0.7487 - loss: 0.4831

2026-01-15 19:55:58.462199: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:55:58.462218: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:55:58.840164: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_1582', 28 bytes spill stores, 28 bytes spill loads



5339/5339 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - f1_score: 0.7491 - loss: 0.4822

2026-01-15 19:56:00.476218: I external/local_xla/xla/service/gpu/autotuning/dot_search_space.cc:208] All configs were filtered out because none of them sufficiently match the hints. Maybe the hints set does not contain a good representative set of valid configs? Working around this by using the full hints set instead.
2026-01-15 19:56:00.632467: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_81', 8 bytes spill stores, 8 bytes spill loads

2026-01-15 19:56:02.220085: I external/local_xla/xla/stream_executor/cuda/subprocess_compilation.cc:346] ptxas warning : Registers are spilled to local memory in function 'gemm_fusion_dot_81', 8 bytes spill stores, 8 bytes spill loads



5339/5339 ━━━━━━━━━━━━━━━━━━━━ 14s 2ms/step - f1_score: 0.8145 - loss: 0.3525 - val_f1_score: 0.6761 - val_loss: 0.1507
Epoch 2/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - f1_score: 0.8669 - loss: 0.2506 - val_f1_score: 0.7358 - val_loss: 0.1204
Epoch 3/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - f1_score: 0.8771 - loss: 0.2381 - val_f1_score: 0.7379 - val_loss: 0.1197
Epoch 4/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - f1_score: 0.8794 - loss: 0.2344 - val_f1_score: 0.7168 - val_loss: 0.1302
Epoch 5/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - f1_score: 0.8812 - loss: 0.2307 - val_f1_score: 0.7509 - val_loss: 0.1176
Epoch 6/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - f1_score: 0.8824 - loss: 0.2306 - val_f1_score: 0.7439 - val_loss: 0.1170
Epoch 7/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step - f1_score: 0.8851 - loss: 0.2291 - val_f1_score: 0.6957 - val_loss: 0.1242
Epoch 8/30
5339/5339 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - f1_score: 0.8845 - loss: 0.2280 - val_f1_sc